# NB05 — Build03 Tools: OpenAI Agents SDK + Regulation Search Tool

## 목적
Build02의 **frozen static RAG**를 더 튜닝하지 않고, 동일한 5-law library를 **agent가 필요할 때 반복 검색할 수 있는 tool**로 바꾼다.

**Build02**  
`question → single retrieval → fixed Top-5 → answer`

**Build03**  
`question → agent 판단 → search_regulation() → observation → 필요 시 refined search → answer`

1차 목표는 **Q7 smoke test**다: Build02에서 fixed Top-5가 놓친 Regulation (EU) 2017/625의 관련 enforcement provisions를 agent가 추가 검색으로 회수할 수 있는지 본다.

### 실험 경계
- 기존 5-law frozen library, legal-aware chunks, benchmark Q1–Q10을 그대로 사용
- retrieval algorithm 자체는 다시 최적화하지 않음
- `Current-law / Version Check Tool`은 Q7 이후 추가
- RASFF search는 이번 Build03 범위 밖


## Notebook map
1. Environment / Agents SDK quickstart  
2. Load frozen Build02 assets  
3. Frozen hybrid retriever를 callable function으로 재사용  
4. `search_regulation()` tool로 wrap  
5. Single Regulatory Agent 생성  
6. Q7 smoke test  
7. tool-call log 저장  
8. Current-law tool placeholder  
9. Q1–Q10 full run은 Q7/Q8 smoke test 이후


In [2]:
# Run once if needed
%pip install -q -U openai-agents sentence-transformers rank-bm25 pandas numpy


Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kubernetes 34.1.0 requires urllib3<2.4.0,>=1.24.2, but you have urllib3 2.8.0 which is incompatible.
langchain-openai 1.0.3 requires openai<3.0.0,>=1.109.1, but you have openai 3.35.1 which is incompatible.


## 1. Environment check
OpenAI Agents SDK의 핵심 구성은 `Agent`, `Runner`, function tool이다.

**중요:** Build02 vs Build03 최종 비교에서는 Build02에서 사용한 것과 **동일한 generation model**을 사용한다.
`OPENAI_MODEL`을 명시적으로 설정하는 방식을 권장한다.


In [1]:
import os, json, re
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

from agents import Agent, Runner
from agents.decorators import tool

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print("OPENAI_API_KEY:", "set" if os.getenv("OPENAI_API_KEY") else "NOT SET")
MODEL_NAME = os.getenv("OPENAI_MODEL", "").strip()
print("OPENAI_MODEL:", MODEL_NAME or "(SDK default; set the Build02 model before final comparison)")


c:\Users\SAMSUNG\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


OPENAI_API_KEY: NOT SET
OPENAI_MODEL: (SDK default; set the Build02 model before final comparison)


### 1A. Tiny function-tool smoke test
먼저 규정 검색을 붙이지 않고, **Agent가 Python function tool을 실제로 호출하는지**만 확인한다.


In [2]:
@tool
def demo_regulation_lookup(topic: str) -> str:
    """Return a tiny demo regulation lookup result.

    Args:
        topic: Food-safety topic to look up.
    """
    if "aflatoxin" in topic.casefold():
        return "Demo result: Regulation (EU) 2023/915 is relevant to maximum contaminant levels."
    return "Demo result: no match."

demo_kwargs = dict(
    name="Build03 Demo Agent",
    instructions="Use demo_regulation_lookup when asked about a food-safety regulation.",
    tools=[demo_regulation_lookup],
)
if MODEL_NAME:
    demo_kwargs["model"] = MODEL_NAME

demo_agent = Agent(**demo_kwargs)

# Uncomment on 10/19
# demo_result = Runner.run_sync(demo_agent, "Which regulation is relevant to aflatoxins?")
# print(demo_result.final_output)


In [4]:
import importlib.metadata as md
import brotli

for package in ["openai-agents", "openai", "httpx2", "Brotli", "brotlicffi"]:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: Not installed")

print("Loaded brotli module:", brotli.__file__)

openai-agents: 0.24.0
openai: 3.35.1
httpx2: 2.13.0
Brotli: 1.0.9
brotlicffi: Not installed
Loaded brotli module: c:\Users\SAMSUNG\anaconda3\Lib\site-packages\brotli.py


In [5]:
%pip install --upgrade "Brotli>=1.2.0"

  Using cached brotli-1.2.0-cp313-cp313-win_amd64.whl.metadata (6.3 kB)
Using cached brotli-1.2.0-cp313-cp313-win_amd64.whl (369 kB)
  Attempting uninstall: Brotli
    Found existing installation: Brotli 1.0.9
    Uninstalling Brotli-1.0.9:
      Successfully uninstalled Brotli-1.0.9
Note: you may need to restart the kernel to use updated packages.


In [1]:

import os
import json
import re
from pathlib import Path
from datetime import datetime, timezone
from getpass import getpass

import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

from agents import Agent, Runner, function_tool

# ============================================================
# 1. Environment setup
# ============================================================

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# Enter your API key securely.
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass(
        "Enter your OpenAI API Key: "
    )

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is required.")

# Requested model
MODEL_NAME = "gpt-6-luna"
os.environ["OPENAI_MODEL"] = MODEL_NAME

print("OPENAI_API_KEY: SET")
print("OPENAI_MODEL:", MODEL_NAME)


# ============================================================
# 2. Define a tiny function tool
# ============================================================

tool_call_log = []

@function_tool
def demo_regulation_lookup(topic: str) -> str:
    """Return a tiny demo regulation lookup result.

    Args:
        topic: Food-safety topic to look up.
    """

    tool_call_log.append(topic)
    print(f"[TOOL EXECUTED] demo_regulation_lookup({topic!r})")

    if "aflatoxin" in topic.casefold():
        return (
            "Demo result: Regulation (EU) 2023/915 "
            "is relevant to maximum contaminant levels."
        )

    return "Demo result: no match."


# ============================================================
# 3. Create a demo agent
# ============================================================

demo_agent = Agent(
    name="Build03 Demo Agent",
    model=MODEL_NAME,
    instructions=(
        "You are a food regulation assistant. "
        "Always use demo_regulation_lookup before answering. "
        "Base your answer on the tool result. "
        "Keep your final answer short."
    ),
    tools=[demo_regulation_lookup],
)

print("Agent created:", demo_agent.name)


# ============================================================
# 4. Run the agent
# ============================================================

# Use await in Jupyter Notebook.
demo_result = await Runner.run(
    demo_agent,
    "Which EU regulation is relevant to aflatoxins?"
)


# ============================================================
# 5. Inspect the result
# ============================================================

print("\n--- FINAL ANSWER ---")
print(demo_result.final_output)

print("\n--- TOOL CALL CHECK ---")
print("Number of tool executions:", len(tool_call_log))
print("Tool inputs:", tool_call_log)

assert len(tool_call_log) >= 1, (
    "The agent did not execute the demo tool."
)

print("\nPASS: Agent successfully executed a function tool.")


c:\Users\SAMSUNG\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


OPENAI_API_KEY: SET
OPENAI_MODEL: gpt-6-luna
Agent created: Build03 Demo Agent
[TOOL EXECUTED] demo_regulation_lookup('aflatoxins EU regulation')

--- FINAL ANSWER ---
Regulation (EU) 2023/915 sets maximum levels for contaminants, including aflatoxins, in food.

--- TOOL CALL CHECK ---
Number of tool executions: 1
Tool inputs: ['aflatoxins EU regulation']

PASS: Agent successfully executed a function tool.


## 2. Load frozen Build02 assets
아래 파일을 그대로 재사용한다.

- `data/chunks_legal_aware.json`
- `data/benchmark_questions_v1.json`
- `data/retrieval_config_nb03_final_hybrid_static_rag.json`

Build02 final config가 바뀌지 않았는지도 assert로 확인한다.


In [2]:
LEGAL_PATH = DATA_DIR / "chunks_legal_aware.json"
BENCHMARK_PATH = DATA_DIR / "benchmark_questions_v1.json"
CONFIG_PATH = DATA_DIR / "retrieval_config_nb03_final_hybrid_static_rag.json"

for p in [LEGAL_PATH, BENCHMARK_PATH, CONFIG_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required Build02 asset: {p}")

legal_chunks = json.loads(LEGAL_PATH.read_text(encoding="utf-8"))
benchmark = json.loads(BENCHMARK_PATH.read_text(encoding="utf-8"))
build02_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
questions = benchmark["questions"]

print("Benchmark:", benchmark.get("benchmark_id"))
print("Chunks:", len(legal_chunks))
print("Method:", build02_config.get("retrieval_method"))
print("RRF k:", build02_config.get("rrf_k"))
print("Candidate depth:", build02_config.get("candidate_depth_per_retriever"))
print("Diversification:", build02_config.get("diversification"))
print("Top-K:", build02_config.get("top_k"))

assert build02_config["retrieval_method"] == "hybrid_dense_bm25_rrf_diverse_v1"
assert build02_config["rrf_k"] == 60
assert build02_config["candidate_depth_per_retriever"] == 50
assert build02_config["diversification"]["max_per_group"] == 2
assert build02_config["top_k"] == 5


Benchmark: regulatory_agent_v1_notion_2026-10-08
Chunks: 1383
Method: hybrid_dense_bm25_rrf_diverse_v1
RRF k: 60
Candidate depth: 50
Diversification: {'group': 'file + article/location', 'max_per_group': 2}
Top-K: 5


## 3. Frozen Build02 retriever → callable function
Build03에서 바뀌는 것은 **검색 알고리즘이 아니라 검색을 사용하는 방식**이다.

Build02 final settings:
- Dense: multilingual MiniLM
- BM25
- RRF `k=60`
- candidate depth = 50 / retriever
- diversification = `file + article/location`, max 2/group

Build02에서는 질문당 한 번만 검색했고, Build03에서는 agent가 같은 retriever를 **여러 번** 호출할 수 있다.


In [3]:
def safe(value):
    return "" if value is None else str(value)

def searchable_text(chunk):
    parts = [
        safe(chunk.get("regulation_title")),
        safe(chunk.get("file")),
        safe(chunk.get("location")),
        safe(chunk.get("article")),
        safe(chunk.get("heading")),
        safe(chunk.get("text")),
    ]
    return " | ".join(p for p in parts if p).strip()

def bm25_tokenize(text):
    return re.findall(r"\b[\w./()-]+\b", text.casefold())

legal_texts = [searchable_text(c) for c in legal_chunks]

EMBEDDING_MODEL = build02_config["embedding_model"]
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

legal_embeddings = embedding_model.encode(
    legal_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

bm25 = BM25Okapi([bm25_tokenize(text) for text in legal_texts])
print("Frozen Build02 retrieval components ready.")


'[WinError 10054] 현재 연결은 원격 호스트에 의해 강제로 끊겼습니다' thrown while requesting HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].
'[WinError 10054] 현재 연결은 원격 호스트에 의해 강제로 끊겼습니다' thrown while requesting HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/adapter_config.json
Retrying in 2s [Retry 2/5].
'[WinError 10054] 현재 연결은 원격 호스트에 의해 강제로 끊겼습니다' thrown while requesting HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/adapter_config.json
Retrying in 4s [Retry 3/5].


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

Frozen Build02 retrieval components ready.


In [4]:
RRF_K = int(build02_config["rrf_k"])
CANDIDATE_DEPTH = int(build02_config["candidate_depth_per_retriever"])
MAX_PER_GROUP = int(build02_config["diversification"]["max_per_group"])

def _group_key(chunk):
    return (
        safe(chunk.get("file")),
        safe(chunk.get("article")) or safe(chunk.get("location"))
    )

def hybrid_search(query: str, top_k: int = 5):
    query = str(query).strip()
    if not query:
        raise ValueError("query must not be empty")

    q_emb = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )[0]

    dense_scores = legal_embeddings @ q_emb
    bm25_scores = np.asarray(bm25.get_scores(bm25_tokenize(query)))

    dense_order = np.argsort(dense_scores)[::-1][:CANDIDATE_DEPTH]
    bm25_order = np.argsort(bm25_scores)[::-1][:CANDIDATE_DEPTH]

    rrf_scores = {}
    for rank, idx in enumerate(dense_order, start=1):
        rrf_scores[int(idx)] = rrf_scores.get(int(idx), 0.0) + 1.0 / (RRF_K + rank)
    for rank, idx in enumerate(bm25_order, start=1):
        rrf_scores[int(idx)] = rrf_scores.get(int(idx), 0.0) + 1.0 / (RRF_K + rank)

    fused_order = sorted(rrf_scores, key=rrf_scores.get, reverse=True)

    selected, group_counts = [], {}
    for idx in fused_order:
        chunk = legal_chunks[idx]
        group = _group_key(chunk)
        if group_counts.get(group, 0) >= MAX_PER_GROUP:
            continue
        selected.append((idx, chunk, rrf_scores[idx]))
        group_counts[group] = group_counts.get(group, 0) + 1
        if len(selected) >= top_k:
            break

    results = []
    for rank, (idx, chunk, rrf_score) in enumerate(selected, start=1):
        results.append({
            "rank": rank,
            "chunk_index": int(idx),
            "rrf_score": float(rrf_score),
            "file": chunk.get("file"),
            "regulation_title": chunk.get("regulation_title"),
            "article": chunk.get("article"),
            "location": chunk.get("location"),
            "heading": chunk.get("heading"),
            "text": chunk.get("text"),
        })
    return results


### 3A. Direct retrieval sanity check
이 cell은 **agent test가 아니다.** retriever function이 동작하는지만 확인한다.


In [5]:
test_results = hybrid_search(
    "What regulatory actions may follow for a non-compliant consignment at EU border control?"
)

for r in test_results:
    print(
        f"[{r['rank']}] {r['file']} | "
        f"{r.get('article') or r.get('location')} | "
        f"{safe(r.get('heading'))[:80]}"
    )


[1] EU_2017_625_consolidated_2025-01-05.html | Article 53 | Official controls not performed at border control posts
[2] EU_2017_625_consolidated_2025-01-05.html | Article 53 | Official controls not performed at border control posts
[3] EU_2017_625_consolidated_2025-01-05.html | Article 51 | Specific rules for official controls at border control posts
[4] EU_2017_625_consolidated_2025-01-05.html | Article 49 | Official controls at border control posts
[5] EU_2017_625_consolidated_2025-01-05.html | Article 49 | Official controls at border control posts


## 4. Wrap as `search_regulation()` tool
Agent는 이 tool을 여러 번 호출할 수 있다.

**중요:** Q7의 target article numbers는 평가 기준일 뿐 agent prompt에 정답으로 주입하지 않는다.


In [7]:
TOOL_CALL_LOG = []

@function_tool
def search_regulation(query: str, top_k: int = 5) -> str:
    """Search the frozen five-law EU food-regulation library.

    If the first search does not provide enough evidence, call this tool again
    with a more focused or complementary query.

    Args:
        query: Focused legal search query.
        top_k: Number of results to return. Default 5.
    """
    top_k = max(1, min(int(top_k), 10))
    results = hybrid_search(query, top_k=top_k)

    TOOL_CALL_LOG.append({
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "query": query,
        "top_k": top_k,
        "results": [
            {
                "rank": r["rank"],
                "file": r["file"],
                "article": r["article"],
                "location": r["location"],
                "heading": r["heading"],
            }
            for r in results
        ],
    })

    return json.dumps([
        {
            "rank": r["rank"],
            "file": r["file"],
            "regulation_title": r["regulation_title"],
            "article": r["article"],
            "location": r["location"],
            "heading": r["heading"],
            "text": r["text"],
        }
        for r in results
    ], ensure_ascii=False, indent=2)

print("Tool ready: search_regulation")


Tool ready: search_regulation


## 5. Single Regulatory Agent
아직 복잡한 planning graph를 만들지 않는다.

Agent의 최소 행동:
1. substantive legal claim은 search tool로 확인
2. 첫 검색이 부족하면 더 좁은 query로 재검색
3. evidence에 없는 세부사항은 만들지 않음
4. frozen snapshot과 current law를 구분


In [8]:
AGENT_INSTRUCTIONS = """
You are a regulatory research agent working from evidence returned by the regulation search tool.

Rules:
1. Use search_regulation for substantive legal claims.
2. If the first search is insufficient, call it again with a narrower or complementary query.
3. Do not invent article numbers, requirements, frequencies, documents, or enforcement actions.
4. Distinguish confirmed evidence from remaining uncertainty.
5. The regulation library is a frozen snapshot; do not claim current applicability solely from this library.
6. Keep the final answer concise and cite regulation number + article/annex where supported.
""".strip()

agent_kwargs = dict(
    name="Build03 Regulatory Research Agent",
    instructions=AGENT_INSTRUCTIONS,
    tools=[search_regulation],
)
if MODEL_NAME:
    agent_kwargs["model"] = MODEL_NAME

regulatory_agent = Agent(**agent_kwargs)
print("Agent ready:", regulatory_agent.name)


Agent ready: Build03 Regulatory Research Agent


In [10]:
print("Build03 model:", regulatory_agent.model)

Build03 model: gpt-6-luna


## 6. Q7 smoke test
Q7에서 관찰할 것:
- tool을 실제 호출했는가?
- 한 번으로 부족할 때 두 번째 검색을 했는가?
- query가 더 구체적으로 바뀌었는가?
- enforcement provisions coverage가 개선됐는가?
- evidence 밖의 조치를 만들지 않았는가?


In [11]:

# ============================================================
# Part 6 — Q7 Smoke Test
# ============================================================

Q7 = questions["Q7"]

print("Q7:", Q7)

# Reset tool-call history
TOOL_CALL_LOG.clear()

# Run the regulatory agent asynchronously in Jupyter
q7_result = await Runner.run(
    regulatory_agent,
    Q7
)

# Final answer
print("\n========== FINAL ANSWER ==========\n")
print(q7_result.final_output)

# Tool execution history
print("\n========== TOOL CALLS ==========\n")
print("Number of tool calls:", len(TOOL_CALL_LOG))

for i, call in enumerate(TOOL_CALL_LOG, start=1):
    print(f"\n--- Tool Call {i} ---")
    print("Query:", call["query"])
    print("Top-K:", call["top_k"])

    for result in call["results"]:
        print(
            f"  [{result['rank']}] "
            f"{result['file']} | "
            f"{result['article'] or result['location']}"
        )

print("\n========== FULL TOOL LOG ==========\n")
print(json.dumps(TOOL_CALL_LOG, ensure_ascii=False, indent=2))



Q7: What regulatory actions may follow if aflatoxin levels exceed the applicable maximum level?

========== FINAL ANSWER ==========

If an aflatoxin result exceeds the applicable maximum level, the food listed in Annex I **must not be placed on the market or used as a food raw material or ingredient** (Regulation (EU) 2023/915, Article 2(1)). The relevant aflatoxin limit depends on the food category (Annex I, section 1.1).

Possible follow-up actions include:

- **Operator action:** If the food has left the operator’s immediate control, the food business operator must initiate withdrawal and inform the competent authority. If it may have reached consumers, the operator must inform them; a consumer recall is required where necessary to protect health (Regulation (EC) No 178/2002, Article 19(1)).
- **Authority action:** For established non-compliance, the competent authority may order measures including restriction or prohibition of placing the food on the market, withdrawal or recall, r

### 6A. Q7 retrieval coverage diagnostic
이 cell은 **agent에게 정답을 알려주기 위한 것이 아니라**, run 이후 evaluation을 위한 diagnostic이다.

Target provisions:
- 2017/625 Art. 66
- Art. 67
- Art. 71
- Art. 72


In [12]:
def q7_target_coverage(tool_call_log):
    joined = json.dumps(tool_call_log, ensure_ascii=False).casefold()
    targets = {
        "Article 66": ["article 66", "art. 66", "art 66"],
        "Article 67": ["article 67", "art. 67", "art 67"],
        "Article 71": ["article 71", "art. 71", "art 71"],
        "Article 72": ["article 72", "art. 72", "art 72"],
    }
    return {
        label: any(pattern in joined for pattern in patterns)
        for label, patterns in targets.items()
    }

# After Q7 run:
coverage = q7_target_coverage(TOOL_CALL_LOG)
print(coverage)
print("Covered:", sum(coverage.values()), "/", len(coverage))


{'Article 66': False, 'Article 67': False, 'Article 71': False, 'Article 72': False}
Covered: 0 / 4


In [13]:

import re

TARGET_ARTICLES = {"66", "67", "71", "72"}

def q7_target_coverage(tool_call_log):
    retrieved = set()

    for call in tool_call_log:
        for result in call["results"]:
            file = str(result.get("file") or "")
            article = str(result.get("article") or "")

            if "2017_625" not in file:
                continue

            match = re.fullmatch(
                r"Article\s+(\d+)",
                article.strip(),
                flags=re.IGNORECASE
            )

            if match:
                retrieved.add(match.group(1))

    return {
        f"Article {a}": a in retrieved
        for a in sorted(TARGET_ARTICLES, key=int)
    }


coverage = q7_target_coverage(TOOL_CALL_LOG)

print(coverage)
print("Covered:", sum(coverage.values()), "/", len(coverage))

print("\nRetrieved 2017/625 provisions:")
for call in TOOL_CALL_LOG:
    for result in call["results"]:
        if "2017_625" in str(result.get("file")):
            print(result.get("article"), "|", result.get("heading"))


{'Article 66': False, 'Article 67': False, 'Article 71': False, 'Article 72': False}
Covered: 0 / 4

Retrieved 2017/625 provisions:
Article 138 | Actions in the event of established non-compliance
Article 138 | Actions in the event of established non-compliance
Article 74 | Non-compliance with, and withdrawal of, the approval of pre-export controls performed by third countries
Article 19 | Specific rules on official controls and for action taken by the competent authorities in relation to the residues of relevant substances in food and feed
Article 19 | Specific rules on official controls and for action taken by the competent authorities in relation to the residues of relevant substances in food and feed
Article 148 | Relation with Regulations (EC) No 852/2004 and (EC) No 853/2004 regarding approval of food business establishments


## 7. Save Q7 smoke-test artifact
답변뿐 아니라 **tool call sequence + retrieved evidence**도 저장한다.


In [15]:
Q7_OUTPUT_PATH = DATA_DIR / "build03_q7_smoke_test.json"

def save_q7_smoke_test(final_output: str):
    payload = {
        "build": "Build03",
        "experiment": "q7_smoke_test",
        "benchmark_id": benchmark.get("benchmark_id"),
        "question_id": "Q7",
        "question": Q7,
        "model": MODEL_NAME or "SDK default",
        "retrieval_method": build02_config["retrieval_method"],
        "retrieval_difference_from_build02": "same frozen retriever exposed as repeatable agent tool",
        "tool_calls": TOOL_CALL_LOG,
        "target_coverage_diagnostic": q7_target_coverage(TOOL_CALL_LOG),
        "final_output": final_output,
        "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    Q7_OUTPUT_PATH.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print("Saved:", Q7_OUTPUT_PATH)

# Example:
save_q7_smoke_test(q7_result.final_output)


Saved: data\build03_q7_smoke_test.json


## 8. Next tool — Current-law / Version Check Tool
**아직 구현하지 않는다. Q7 smoke test가 먼저다.**

예상 interface:

```python
check_current_law(
    regulation_number: str,
    as_of_date: str | None = None
)
```

역할:
- official EU source의 current consolidated version 확인
- amendment / consolidation date 확인
- frozen local snapshot과 current source 구분

설계 원칙:
- `search_regulation()` = frozen evidence retrieval
- `check_current_law()` = currentness verification


## 9. Full Q1–Q10 run — later
아래가 끝난 뒤에만 전체 실행:
- [ ] Agents SDK basic tool call 성공
- [ ] Regulation Search Tool 정상 작동
- [ ] Q7 repeat search behavior 확인
- [ ] Q7 evidence coverage 기록
- [ ] Current-law / Version Check Tool 구현
- [ ] Q8 currentness smoke test 완료

그 다음 Build02와 동일한 frozen checklist:
- Correct
- Grounded
- Complete

추가 Build03 diagnostic:
- tool called?
- number of tool calls
- query sequence
- evidence coverage
- unsupported claim
- currentness check used?


# 10/19 시작 체크리스트
첫날에는 여기까지만 성공해도 충분하다.

- [ ] `openai-agents` 설치
- [ ] API key 확인
- [ ] demo tool을 agent가 실제 호출
- [ ] Build02 assets load
- [ ] `hybrid_search()` direct test
- [ ] `search_regulation()` tool 생성
- [ ] Regulatory Agent 생성

핵심 질문:

> **더 좋은 retriever를 만들었는가?**가 아니라  
> **같은 retriever를 agent가 필요할 때 다시 사용하면 Build02의 coverage failure를 줄일 수 있는가?**
